# Error Analysis: Top Discrepancies (Russian & English)

This notebook analyzes the `error_analysis_merged.jsonl` file to identify the most significant errors in the model's predictions for English and Russian subsets.

In [1]:
import pandas as pd
import json
import os

# 1. Load the Data
data_path = "/kaggle/working/error_analysis_merged.jsonl"
print(f"Reading data from: {data_path}")

records = []
with open(data_path, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

df = pd.DataFrame(records)
print(f"Total records loaded: {len(df)}")

Reading data from: /kaggle/working/error_analysis_merged.jsonl
Total records loaded: 16186


In [2]:
# 2. Enhance Data

# Extract Language from Source_File (e.g., 'checkpoint-2520_eng_laptop_test_task1.json')
def get_lang(filename):
    filename = str(filename).lower()
    if "eng_" in filename: return "English"
    if "rus_" in filename: return "Russian"
    return "Other"

df['Language'] = df['Source_File'].apply(get_lang)

# Calculate a combined error metric (Sum of Absolute Errors)
df['Total_Error_Abs'] = df['Abs_Error_Valence'] + df['Abs_Error_Arousal']

print("Distribution by Language:")
print(df['Language'].value_counts())

Distribution by Language:
Language
Other      11624
English     2925
Russian     1637
Name: count, dtype: int64


In [3]:
# 3. Filter for Target Languages (English & Russian)
target_langs = ['English', 'Russian']
df_filtered = df[df['Language'].isin(target_langs)].copy()

print(f"Records after filtering for {target_langs}: {len(df_filtered)}")

Records after filtering for ['English', 'Russian']: 4562


In [4]:
# 4. Get Top 100 "Most Egregious" Errors
# Sorted by Total Absolute Error (Valence Error + Arousal Error)

top_100 = df_filtered.sort_values(by='Total_Error_Abs', ascending=False).head(100)

# Select Columns for clean display
display_cols = [
    'ID',
    'Language',
    'Source_File',
    'Text',
    'Target',
    'Gold_Valence', 'Predicted_Valence', 'Error_Valence',
    'Gold_Arousal', 'Predicted_Arousal', 'Error_Arousal',
    'Total_Error_Abs'
]

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 100)

print("Top 100 Errors (Sorted by Sum of Absolute Errors):")
display(top_100[display_cols])

Top 100 Errors (Sorted by Sum of Absolute Errors):


,ID,Language,Source_File,Text,Target,Gold_Valence,Predicted_Valence,Error_Valence,Gold_Arousal,Predicted_Arousal,Error_Arousal,Total_Error_Abs
15140,12582:4_373,Russian,checkpoint-2520_rus_restaurant_test_task1.json,"Кухня в ЗимаЛето очень неплохая: хорошее и вкусное гриль-меню (особенно рыба), стандартный выбор салатиков, большой выбор суши и роллов (самые вкусные Гейша и запеченные Киото), достойная коктельная карта, только вот лонг, оставляет желать лучшего, а клубничную маргариту не советую никому, отвратная!",клубничную маргариту,1.17,7.10,5.93,8.84,6.35,-2.49,8.42
15021,11817:1_322,Russian,checkpoint-2520_rus_restaurant_test_task1.json,"Пригляделась, оказалось открылся ресторанчик ""Палки"" японской и итальянской кухни, ну, думаю, зайду ...лучше бы и не заходила.","ресторанчик ""Палки""",1.50,7.15,5.65,8.50,6.56,-1.94,7.59
1235,lap26_aspect_va_test_873,English,checkpoint-2520_eng_laptop_test_task1.json,"it was a great laptop if I just wanted to stare at the desktop ( it lagged even bringing up the control panel ) , so I took it back immediately and got this",laptop,3.00,7.41,4.41,4.25,7.29,3.04,7.45
15887,34607:6_893,Russian,checkpoint-2520_rus_restaurant_test_task1.json,"Рис також чекали разом з чаєм більше 20 хв., але я то думаю,китайський ресторан,як на сході оформляють страви ми всі добре знаємо,завжди все прикрашено і виглядає дуже апетитно,так ось,нам рис принесли на тарілці в такому вигляді,що я навіть їсти не став його.",рис,1.00,5.75,4.75,8.75,6.14,-2.61,7.36
15824,34228:2_857,Russian,checkpoint-2520_rus_restaurant_test_task1.json,Ресторандагы иң яхшы зал тәмәке тартучыларныкы итеп эшләнгән.,зал,3.25,8.26,5.01,5.75,7.95,2.20,7.21
15439,15847:12_587,Russian,checkpoint-2520_rus_restaurant_test_task1.json,"Туда если только за дешевыми акциями ходить, но чего то карту покупать не хочется, все же ждешь положительных эмоций от посещения подобных заведений за любые деньги.",заведений,1.75,6.79,5.04,8.25,6.33,-1.92,6.96
8692,rest26_aspect_va_test_9,English,checkpoint-2520_eng_restaurant_test_task1.json,"Peppermill , if you want to create a cool cocktail bar , ditch the machines , train your staff , or hire bartenders with better service skills , and buy some independent liquors",Peppermill,1.50,6.02,4.52,8.00,5.61,-2.39,6.91
15436,15847:1_584,Russian,checkpoint-2520_rus_restaurant_test_task1.json,"Егет белән кердек, безгә бер официант та карамады!",официант,1.25,5.95,4.70,8.75,6.57,-2.18,6.88
814,lap26_aspect_va_test_570,English,checkpoint-2520_eng_laptop_test_task1.json,but even my now crappy ASUS works better than this brand new Chromebook,brand new Chromebook,1.50,7.45,5.95,8.38,7.48,-0.90,6.85
9345,rest26_aspect_va_test_457,English,checkpoint-2520_eng_restaurant_test_task1.json,Your better off going to the fast food chicken joint across the street El Pollo Loco and guessing the kitchen is cleaner and the staff is friendlier and you won't be subjected to the daily drunks that drive to and from this place daily,this place,1.33,5.95,4.62,7.83,5.61,-2.22,6.84


In [5]:
# 5. Save to CSV for download if needed
top_100[display_cols].to_csv("top_100_errors_rus_eng.csv", index=False)
print("Saved top 100 errors to 'top_100_errors_rus_eng.csv'")

Saved top 100 errors to 'top_100_errors_rus_eng.csv'
